# 15_retrain_controls.ipynb

Task 5.9 và 5.10: huấn luyện lại Swin-Tiny với bốn kiểu augmentation theo **một giao thức chung**, nhiều seed, để trả lời lại câu hỏi của TN7 ("phần cải thiện của huấn luyện lại đến từ augmentation nói chung hay từ việc nhắm đúng họ hỏng") sau khi sửa các điểm chưa chặt: một seed, ngân sách epoch lệch (baseline dừng ở epoch 12, hai nhánh kia chạy 30), quy tắc chọn checkpoint lệch (baseline chọn trên validation sạch).

Giao thức (khóa trong `configs/controls/protocol.yaml`, không chỉnh theo kết quả): 30 epoch, không dừng sớm, chọn checkpoint bằng macro F1 trên validation hỗn hợp (ảnh sạch cộng 14 họ ở mức 2), cùng optimizer và seed 42, 43, 44.

| Task | Nhánh | Seed | Ghi chú |
|---|---|---|---|
| 5.9 | baseline (chỉ lật, xoay) | 42, 43, 44 | train lại cả ba |
| 5.9 | suy giảm ngẫu nhiên (4.5) | 42, 43, 44 | seed 42 dùng lại từ lần chạy cũ nếu khớp giao thức, train 43 và 44 |
| 5.9 | TrivialAugment Wide (5.6) | 42, 43, 44 | như trên |
| 5.10 | TrivialAugment chỉ phép hình học | 42 | cần 5.9 để so |
| 5.10 | suy giảm bỏ nhóm họ chọn bằng MSD (leave-group-out) | 42 | cần `reports/unseen_v2/msd.json` của task 5.7 |

Chỉ dùng train và validation. **Tập test không mở.** Kết quả ở đây là validation, một bước trung gian. Đo cuối trên bộ họ mới và test thuộc 5.11 và P6.1.

## 1. Môi trường

Internet On, Accelerator GPU (T4). Input bắt buộc: `cthng123/durian-leaf-disease-dataset`. Mỗi lần train khoảng 40 đến 45 phút. Kaggle giới hạn phiên 9 giờ, nên chia ra nhiều phiên (biến `STAGE` và `SEEDS` ở mục 2) và thêm output của phiên trước làm input để notebook bỏ qua phần đã xong.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Đổi nhánh tại đây, rồi chạy lại cell.
GIT_BRANCH = "retrain-controls-5.6-5.9-5.10"
REPO_URL = "https://github.com/meo225/durian-leaf-disease-robustness.git"
REPO_DIR = Path("/kaggle/working") / "durian-leaf-disease-robustness"

if not (Path("/kaggle/working").is_dir() and Path("/kaggle/input").is_dir()):
    raise RuntimeError("Notebook này chỉ chạy trên Kaggle. Xem README, mục Chạy trên Kaggle.")

def git(args, cwd=None):
    subprocess.run(
        ["git", *args],
        cwd=cwd,
        check=True,
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
    )


if not (REPO_DIR / ".git").is_dir():
    git(["clone", "--branch", GIT_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
else:
    git(["fetch", "origin", GIT_BRANCH], cwd=REPO_DIR)
    git(["checkout", GIT_BRANCH], cwd=REPO_DIR)
    git(["reset", "--hard", f"origin/{GIT_BRANCH}"], cwd=REPO_DIR)

matches = []
seen = set()
for path in Path("/kaggle/input").rglob("DLD_FinalDataset_224_spit"):
    resolved = str(path.resolve())
    if path.is_dir() and resolved not in seen:
        seen.add(resolved)
        matches.append(path)

if not matches:
    found = []
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        for path in sorted(input_root.rglob("*")):
            if path.is_dir() and len(path.relative_to(input_root).parts) <= 3:
                found.append(str(path.relative_to(input_root)))
    listing = "\n".join(found[:40]) if found else "(trống)"
    raise FileNotFoundError(
        "Chưa thấy DLD_FinalDataset_224_spit trong /kaggle/input.\n"
        "Add Input: cthng123/durian-leaf-disease-dataset, rồi Save & Run All.\n"
        f"Thư mục hiện có:\n{listing}"
    )

image_root = matches[0].parent
link = REPO_DIR / "datasets" / "durian-ldd"
link.parent.mkdir(parents=True, exist_ok=True)
if link.is_symlink() or link.is_file():
    link.unlink()
elif link.exists():
    link.rmdir()
link.symlink_to(image_root, target_is_directory=True)

if not (link / "DLD_FinalDataset_224_spit").is_dir():
    raise FileNotFoundError(f"Không thấy ảnh tại {link}")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR, text=True
).strip()
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=REPO_DIR, text=True
).strip()
print(f"Nhánh: {branch}")
print(f"Commit: {commit}")
print(f"Ảnh: {link / 'DLD_FinalDataset_224_spit'}")

## 2. Chọn lần chạy

Đổi `STAGE` rồi chạy từ cell này xuống.

- `smoke`: chỉ baseline seed 42. **Chạy cái này đầu tiên** để bắt lỗi cấu hình trước khi tốn giờ GPU cho các lần còn lại.
- `5.9`: ba nhánh, ba seed. Seed 42 của suy giảm và TrivialAugment dùng lại từ lần chạy cũ nếu JSON cũ khớp giao thức, nên còn 7 lần train.
- `5.10`: TrivialAugment hình học và suy giảm bỏ nhóm họ. Cần `reports/unseen_v2/msd.json` (task 5.7) trên nhánh, và nên đã có kết quả 5.9 để so.

`SEEDS = (43,)` chỉ chạy seed 43, để chia nhiều phiên. Chạy lại cell này sau khi phiên đứt thì phần đã xong được bỏ qua.

In [ ]:
import json
import shutil
import zipfile

import pandas as pd
from IPython.display import display

from src.evaluation import seed_summary as S
from src.training import run_control as C
from src.training.run import ROOT

STAGE = "smoke"  # "smoke" | "5.9" | "5.10"
SEEDS = None  # None = mọi seed của giai đoạn, hoặc ví dụ (43,)

PLANS = {"smoke": (C.SMOKE,), "5.9": C.PLAN_5_9, "5.10": C.PLAN_5_10}
plan = [spec for spec in PLANS[STAGE] if SEEDS is None or spec.seed in SEEDS]
RES = ROOT / "outputs/results"
REP = REPO_DIR / "reports/controls"
REP.mkdir(parents=True, exist_ok=True)

copied = C.import_previous_results("/kaggle/input", RES)
print(f"Chép từ output phiên trước: {len(copied)} file")

status = []
for spec in plan:
    found = C.find_run(spec, RES)
    status.append({"nhánh": spec.arm, "seed": spec.seed, "trạng thái": "cần train" if found is None else {"trained": "đã có (train mới)", "reused": "dùng lại lần chạy cũ"}[found["source"]]})
display(pd.DataFrame(status))
print(f"Cần train: {sum(1 for item in status if item['trạng thái'] == 'cần train')} lần")

## 3. Train

Mỗi lần train in loss và macro F1 từng epoch (dòng có chữ `checkpoint` là epoch đang giữ), rồi đo ảnh sạch và 42 điều kiện đã thấy trên validation. Khoảng 40 đến 45 phút một lần. Thiếu GPU thì dừng ngay.

In [ ]:
trained = C.run_plan(plan, RES)
print("Vừa train:", [(spec.arm, spec.seed) for spec in trained] or "(không có, tất cả đã xong)")

## 4. Kết quả từng lần chạy

`mixed_val_f1` là điểm dùng để chọn checkpoint (validation hỗn hợp, chứa chính các họ đang đo), nên là điểm chọn, không phải điểm báo cáo. `objective_42` là macro F1 trung bình 42 điều kiện đã thấy trên validation (cùng thước đo của 4.5 và 5.6). `source = reused` là lần chạy cũ đã qua kiểm giao thức.

In [ ]:
nine = C.run_table(C.PLAN_5_9, RES)
display(nine.round(4))
missing = [(spec.arm, spec.seed) for spec in C.PLAN_5_9 if not ((nine["arm"] == spec.arm) & (nine["seed"] == spec.seed)).any()]
print("Chưa có:", missing or "(đủ 9 lần chạy)")

## 5. Task 5.9: ba nhánh, nhiều seed

Quy tắc kết luận khai báo trước: chỉ kết luận khi chênh lệch trung bình lớn hơn 2,5 lần độ lệch chuẩn seed (SD mẫu, gộp hai nhánh). Với 3 seed SD tự nó nhiễu, nên đây là mức tối thiểu để dám nói. Chênh lệch nhỏ hơn mức đó ghi "chưa kết luận", không đọc thành "bằng nhau".

In [ ]:
ready = [arm for arm in ("baseline", "degradation", "trivialaugment") if (nine["arm"] == arm).sum() >= 2]
if len(ready) < 3:
    print("Chưa đủ nhánh có từ 2 seed trở lên để so. Chạy tiếp các seed còn thiếu.")
else:
    values = {arm: list(nine.loc[nine["arm"] == arm, "objective_42"]) for arm in ready}
    stats = pd.DataFrame({arm: S.seed_stats(items) for arm, items in values.items()}).T
    display(stats.round(4))
    comparison = S.compare_arms(values, S.PAIRS_5_9)
    display(comparison.round(4))
    print("Số seed mỗi nhánh:", {arm: len(items) for arm, items in values.items()}, "(kết luận chỉ có giá trị khi đủ 3)")
    clean_values = {arm: list(nine.loc[nine["arm"] == arm, "clean_val_f1"]) for arm in ready}
    display(pd.DataFrame({arm: S.seed_stats(items) for arm, items in clean_values.items()}).T.round(4))

## 6. Task 5.10: TrivialAugment hình học và leave-group-out

Hai nhánh một seed (42), nên không có SD riêng. Độ lệch chuẩn lấy từ nhánh tham chiếu 3 seed của 5.9, và quy tắc 2,5 lần SD áp như trên. Đọc kết quả theo bảng khai báo trước trong `docs/task-5.10-ta-geometric-and-leave-group.md`. Chạy cell này sau khi đã có 5.10 và 5.9.

Với nhánh leave-group, nhóm họ bị bỏ ghi trong JSON (`leave_group.families`). `objective_group` là điểm trên đúng các họ đó (chưa từng thấy lúc train), `objective_rest` là điểm trên các họ còn lại.

In [ ]:
ten = C.run_table(C.PLAN_5_10, RES)
if ten.empty:
    print("Chưa có kết quả 5.10.")
else:
    leave = C.find_run(C.RunSpec("degradation_leave_group", 42), RES)
    group = json.loads(leave["record"].read_text(encoding="utf-8"))["leave_group"] if leave else None
    if group:
        print("Nhóm họ bị bỏ (chọn bằng MSD):", group["families"])
        print(f"Khoảng cách trong nhóm {group['mean_distance_inside']:.3f}, ra ngoài nhóm {group['mean_distance_to_rest']:.3f}")
    everything = C.run_table([*C.PLAN_5_9, *C.PLAN_5_10], RES, group_families=group["families"] if group else None)
    display(everything.round(4))
    references = {arm: everything.loc[everything["arm"] == arm] for arm in ("baseline", "degradation", "trivialaugment")}
    rows = []
    for arm in ("trivialaugment_geometric", "degradation_leave_group"):
        single = everything.loc[everything["arm"] == arm]
        if single.empty:
            continue
        for reference, frame in references.items():
            if len(frame) < 2:
                continue
            columns = ["objective_42"] + (["objective_group", "objective_rest"] if arm == "degradation_leave_group" and group else [])
            for column in columns:
                rows.append({"nhánh": arm, "so với": reference, "thước đo": column, "giá trị": float(single[column].iloc[0]), **S.compare_single_to_arm(float(single[column].iloc[0]), list(frame[column]))})
    comparison_ten = pd.DataFrame(rows)
    display(comparison_ten.round(4))

## 7. Lưu kết quả

Chép JSON, bảng điều kiện và các bảng tổng hợp vào `reports/controls/` (bản chốt để commit), rồi nén. Checkpoint `.pt` nằm ở tab Output, trong `outputs/checkpoints/`. Checkpoint không vào Git, nhưng 5.11 cần chúng: sau khi xong, lưu phiên bằng **Save Version** rồi đưa output thành dataset Kaggle.

In [ ]:
produced = [spec for spec in plan if (found := C.find_run(spec, RES)) is not None and found["source"] == "trained"]
for spec in produced:
    train_cfg, _ = C.load_arm_config(spec.config)
    for source in RES.glob(f"{train_cfg['model_name']}_seed{spec.seed}*"):
        shutil.copy(source, REP / source.name)
if not nine.empty:
    nine.to_csv(REP / "runs_5_9.csv", index=False)
if len(ready) == 3:
    stats.to_csv(REP / "seed_stats_5_9.csv")
    comparison.to_csv(REP / "comparison_5_9.csv", index=False)
if not ten.empty:
    everything.to_csv(REP / "runs_5_9_5_10.csv", index=False)
    comparison_ten.to_csv(REP / "comparison_5_10.csv", index=False)

archive = Path("/kaggle/working/retrain_controls_outputs.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(REP.rglob("*")):
        if path.is_file():
            bundle.write(path, path.relative_to(REPO_DIR).as_posix())
print(f"Đã nén {archive.name}. Tải về, giải nén vào repo (thư mục reports/controls/), rồi commit.")